In [1]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sqlalchemy import create_engine

engine = create_engine('mysql+pymysql://root:@localhost/dw_agriculture')

# Charger les tables
fait = pd.read_sql("SELECT * FROM fait_meteo", engine)
dim_station = pd.read_sql("SELECT * FROM dim_station", engine)
dim_temps = pd.read_sql("SELECT * FROM dim_temps", engine)
dim_alerte = pd.read_sql("SELECT * FROM dim_alerte", engine)

# Jointures
df = fait.merge(dim_station, on='id_station', how='left')
df = df.merge(dim_temps, on='id_date', how='left')
df = df.merge(dim_alerte, on='id_alerte', how='left')

print("✅ Données chargées :", df.shape)
print(df.columns.tolist())

✅ Données chargées : (4626, 24)
['id_fait', 'id_date', 'id_station', 'id_alerte', 'temp_c', 'hum_pct', 'wind_speed', 'precip_mm', 'severity_index_x', 'indice_risque', 'nom_station', 'ville', 'altitude', 'zone_geo', 'capteur_type', 'date', 'jour', 'mois', 'trimestre', 'annee', 'saison', 'severity_index_y', 'alert_msg', 'type_precip']


In [6]:
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Theme global
THEME = {
    'bg': '#0f1117',
    'card': '#1a1d2e',
    'accent': '#00d4ff',
    'green': '#00ff88',
    'red': '#ff4757',
    'orange': '#ffa502',
    'yellow': '#ffdd59',
    'text': '#ffffff',
    'subtext': '#8892b0'
}

# ============================================
# KPI CARDS stylées
# ============================================
fig_kpi = go.Figure()

kpis = [
    {'label': '📍 Stations Actives', 'value': stations_actives, 'color': THEME['accent']},
    {'label': '📋 Total Relevés', 'value': total_releves, 'color': THEME['green']},
    {'label': '🌡️ Temp Max (°C)', 'value': temp_max, 'color': THEME['orange']},
    {'label': '🚨 Alertes Rouge', 'value': alertes_rouge, 'color': THEME['red']},
    {'label': '⚠️ Indice Risque Moyen', 'value': indice_moyen, 'color': THEME['yellow']},
]

for i, kpi in enumerate(kpis):
    fig_kpi.add_trace(go.Indicator(
        mode="number",
        value=kpi['value'],
        title={'text': kpi['label'], 'font': {'size': 14, 'color': THEME['subtext']}},
        number={'font': {'size': 36, 'color': kpi['color']}},
        domain={'x': [i/5, (i+1)/5], 'y': [0, 1]}
    ))

fig_kpi.update_layout(
    title={
        'text': '🌍 Agriculture-Résilience 2030 — Tableau de Bord Stratégique',
        'font': {'size': 22, 'color': THEME['text']},
        'x': 0.5
    },
    paper_bgcolor=THEME['bg'],
    plot_bgcolor=THEME['bg'],
    height=200,
    margin=dict(t=60, b=10)
)
fig_kpi.show()

# ============================================
# GRAPHIQUE 1 — Température par zone (horizontal)
# ============================================
temp_zone = df.groupby('zone_geo')['temp_c'].mean().round(1).reset_index()
temp_zone.columns = ['Zone', 'Temp_Moyenne']
temp_zone = temp_zone.sort_values('Temp_Moyenne', ascending=True)

fig1 = go.Figure(go.Bar(
    x=temp_zone['Temp_Moyenne'],
    y=temp_zone['Zone'],
    orientation='h',
    marker=dict(
        color=temp_zone['Temp_Moyenne'],
        colorscale='RdYlGn_r',
        showscale=True,
        colorbar=dict(title='°C', tickfont=dict(color=THEME['text']))
    ),
    text=temp_zone['Temp_Moyenne'].astype(str) + '°C',
    textposition='outside',
    textfont=dict(color=THEME['text'])
))

fig1.update_layout(
    title={'text': '🌡️ Température Moyenne par Zone Géographique',
           'font': {'size': 16, 'color': THEME['text']}, 'x': 0.5},
    paper_bgcolor=THEME['bg'],
    plot_bgcolor=THEME['card'],
    height=450,
    xaxis=dict(gridcolor='#2d3561', color=THEME['subtext'], title='Température (°C)'),
    yaxis=dict(gridcolor='#2d3561', color=THEME['text']),
    margin=dict(l=20, r=20, t=50, b=20)
)
fig1.show()

# ============================================
# GRAPHIQUE 2 — Précipitations par mois
# ============================================
precip_mois = df.groupby('mois')['precip_mm'].sum().reset_index()
mois_labels = {1:'Jan', 2:'Fév', 3:'Mar', 4:'Avr', 5:'Mai', 6:'Jun',
               7:'Jul', 8:'Aoû', 9:'Sep', 10:'Oct', 11:'Nov', 12:'Déc'}
precip_mois['Mois_Label'] = precip_mois['mois'].map(mois_labels)

fig2 = go.Figure(go.Bar(
    x=precip_mois['Mois_Label'],
    y=precip_mois['precip_mm'],
    marker=dict(
        color=precip_mois['precip_mm'],
        colorscale='Blues',
        showscale=False,
        line=dict(color=THEME['accent'], width=1)
    ),
    text=precip_mois['precip_mm'].round(0).astype(int).astype(str) + 'mm',
    textposition='outside',
    textfont=dict(color=THEME['text'])
))

fig2.update_layout(
    title={'text': '🌧️ Précipitations Totales par Mois',
           'font': {'size': 16, 'color': THEME['text']}, 'x': 0.5},
    paper_bgcolor=THEME['bg'],
    plot_bgcolor=THEME['card'],
    height=400,
    xaxis=dict(gridcolor='#2d3561', color=THEME['text']),
    yaxis=dict(gridcolor='#2d3561', color=THEME['subtext'], title='mm'),
    margin=dict(l=20, r=20, t=50, b=20)
)
fig2.show()

# ============================================
# GRAPHIQUE 3 — Alertes (donut)
# ============================================
alertes = df['severity_index'].value_counts().reset_index()
alertes.columns = ['Severity', 'Count']

color_map = {
    'Rouge': THEME['red'],
    'Orange': THEME['orange'],
    'Jaune': THEME['yellow'],
    'RAS': THEME['green']
}

fig3 = go.Figure(go.Pie(
    labels=alertes['Severity'],
    values=alertes['Count'],
    hole=0.6,
    marker=dict(
        colors=[color_map.get(s, '#888') for s in alertes['Severity']],
        line=dict(color=THEME['bg'], width=3)
    ),
    textfont=dict(color=THEME['text'], size=13),
    hovertemplate='<b>%{label}</b><br>%{value} relevés<br>%{percent}<extra></extra>'
))

fig3.add_annotation(
    text=f"<b>{len(df)}</b><br>Relevés",
    x=0.5, y=0.5,
    font=dict(size=18, color=THEME['text']),
    showarrow=False
)

fig3.update_layout(
    title={'text': '🚨 Répartition des Alertes Météo',
           'font': {'size': 16, 'color': THEME['text']}, 'x': 0.5},
    paper_bgcolor=THEME['bg'],
    plot_bgcolor=THEME['bg'],
    height=400,
    legend=dict(font=dict(color=THEME['text']), bgcolor=THEME['card']),
    margin=dict(l=20, r=20, t=50, b=20)
)
fig3.show()

# ============================================
# GRAPHIQUE 4 — Indice risque par zone
# ============================================
risque_zone = df.groupby('zone_geo')['indice_risque'].mean().round(2).reset_index()
risque_zone.columns = ['Zone', 'Indice_Risque']
risque_zone = risque_zone.sort_values('Indice_Risque', ascending=False)

colors_risque = ['#ff4757' if v > 60 else '#ffa502' if v > 45 else '#00ff88'
                 for v in risque_zone['Indice_Risque']]

fig4 = go.Figure(go.Bar(
    x=risque_zone['Zone'],
    y=risque_zone['Indice_Risque'],
    marker=dict(color=colors_risque, line=dict(color=THEME['bg'], width=1)),
    text=risque_zone['Indice_Risque'],
    texttemplate='%{text}',
    textposition='outside',
    textfont=dict(color=THEME['text'])
))

fig4.add_hline(y=60, line_dash='dash', line_color=THEME['red'],
               annotation_text='Seuil Critique', annotation_font_color=THEME['red'])
fig4.add_hline(y=45, line_dash='dash', line_color=THEME['orange'],
               annotation_text='Seuil Alerte', annotation_font_color=THEME['orange'])

fig4.update_layout(
    title={'text': '⚠️ Indice de Risque par Zone (Rouge>60 | Orange>45 | Vert<45)',
           'font': {'size': 16, 'color': THEME['text']}, 'x': 0.5},
    paper_bgcolor=THEME['bg'],
    plot_bgcolor=THEME['card'],
    height=450,
    xaxis=dict(gridcolor='#2d3561', color=THEME['text'], tickangle=-30),
    yaxis=dict(gridcolor='#2d3561', color=THEME['subtext'], title='Indice Risque'),
    margin=dict(l=20, r=20, t=60, b=60)
)
fig4.show()

# ============================================
# GRAPHIQUE 5 — Evolution températures
# ============================================
temp_mois = df.groupby('mois').agg(
    Temp_Max=('temp_c', 'max'),
    Temp_Min=('temp_c', 'min'),
    Temp_Moy=('temp_c', 'mean')
).round(1).reset_index()
temp_mois['Mois_Label'] = temp_mois['mois'].map(mois_labels)

fig5 = go.Figure()

fig5.add_trace(go.Scatter(
    x=temp_mois['Mois_Label'], y=temp_mois['Temp_Max'],
    name='Maximum', line=dict(color=THEME['red'], width=2.5),
    fill=None, mode='lines+markers',
    marker=dict(size=6, color=THEME['red'])
))

fig5.add_trace(go.Scatter(
    x=temp_mois['Mois_Label'], y=temp_mois['Temp_Moy'],
    name='Moyenne', line=dict(color=THEME['accent'], width=2.5),
    fill='tonexty', fillcolor='rgba(255,71,87,0.1)',
    mode='lines+markers', marker=dict(size=6, color=THEME['accent'])
))

fig5.add_trace(go.Scatter(
    x=temp_mois['Mois_Label'], y=temp_mois['Temp_Min'],
    name='Minimum', line=dict(color=THEME['green'], width=2.5),
    fill='tonexty', fillcolor='rgba(0,212,255,0.1)',
    mode='lines+markers', marker=dict(size=6, color=THEME['green'])
))

fig5.update_layout(
    title={'text': '📈 Évolution des Températures par Mois',
           'font': {'size': 16, 'color': THEME['text']}, 'x': 0.5},
    paper_bgcolor=THEME['bg'],
    plot_bgcolor=THEME['card'],
    height=400,
    xaxis=dict(gridcolor='#2d3561', color=THEME['text']),
    yaxis=dict(gridcolor='#2d3561', color=THEME['subtext'], title='°C'),
    legend=dict(font=dict(color=THEME['text']), bgcolor=THEME['card']),
    margin=dict(l=20, r=20, t=50, b=20)
)
fig5.show()

# ============================================
# GRAPHIQUE 6 — Scatter Humidité vs Risque
# ============================================
sample = df.sample(500)

fig6 = px.scatter(
    sample,
    x='hum_pct', y='indice_risque',
    color='zone_geo',
    size='wind_speed',
    hover_data=['ville', 'temp_c'],
    title='💧 Humidité vs Indice de Risque (taille = vent)',
    labels={'hum_pct': 'Humidité (%)', 'indice_risque': 'Indice de Risque', 'zone_geo': 'Zone'},
    opacity=0.8,
    color_discrete_sequence=px.colors.qualitative.Vivid
)

fig6.update_layout(
    paper_bgcolor=THEME['bg'],
    plot_bgcolor=THEME['card'],
    height=450,
    font=dict(color=THEME['text']),
    xaxis=dict(gridcolor='#2d3561', color=THEME['text']),
    yaxis=dict(gridcolor='#2d3561', color=THEME['subtext']),
    legend=dict(font=dict(color=THEME['text']), bgcolor=THEME['card']),
    margin=dict(l=20, r=20, t=50, b=20),
    title={'font': {'size': 16, 'color': THEME['text']}, 'x': 0.5}
)
fig6.show()

print("\n🎉 Dashboard stylé complet !")


🎉 Dashboard stylé complet !


In [7]:
# Renommer severity_index_x en severity_index
df = df.rename(columns={'severity_index_x': 'severity_index'})
df = df.drop(columns=['severity_index_y'])

print("✅ Colonnes corrigées")
print(df.columns.tolist())

KeyError: "['severity_index_y'] not found in axis"